<a href="https://colab.research.google.com/github/RemyaDeepesh/executive_program_AI_ML_python_pgm/blob/main/DeepLearning_Titanic_dataset_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# IMPLEMENTING A SIMPLE NEURAL NETWORK ON THE TITANIC DATASET


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [2]:
file_path='/content/drive/MyDrive/ExecutiveProgramInAdvanced-AI ML-IIT Palakkad-Feb2026/titanic_data.csv'
df = pd.read_csv(file_path)

In [3]:
print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())

Dataset Shape: (893, 12)

First 5 Rows:
   PassengerId  Survived  Pclass  \
0          1.0       0.0     3.0   
1          2.0       1.0     1.0   
2          3.0       1.0     3.0   
3          4.0       1.0     1.0   
4          5.0       0.0     3.0   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0    1.0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0    1.0   
2                             Heikkinen, Miss. Laina  female  26.0    0.0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0    1.0   
4                           Allen, Mr. William Henry    male  35.0    0.0   

   Parch            Ticket     Fare Cabin Embarked  
0    0.0         A/5 21171   7.2500   NaN        S  
1    0.0          PC 17599  71.2833   C85        C  
2    0.0  STON/O2. 3101282   7.9250   NaN        S  
3    0.0            113803  53.1000  C123        S  
4    0.0       

In [4]:
print("\nDataset Info:")
print(df.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893 entries, 0 to 892
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    float64
 1   Survived     891 non-null    float64
 2   Pclass       891 non-null    float64
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    float64
 7   Parch        891 non-null    float64
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(7), object(5)
memory usage: 83.8+ KB
None


In [5]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
PassengerId      2
Survived         2
Pclass           2
Name             2
Sex              2
Age            179
SibSp            2
Parch            2
Ticket           2
Fare             2
Cabin          689
Embarked         4
dtype: int64


##DATA PREPROCESSING

In [6]:
# Drop rows where ALL values are NaN (empty rows)
df.dropna(how='all', inplace=True)

In [7]:
# Drop rows where target variable (Survived) is NaN
df.dropna(subset=['Survived'], inplace=True)

In [8]:
# Drop irrelevant columns
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

In [9]:
# Drop any remaining rows with NaN in critical columns
df.dropna(subset=['Pclass', 'Sex', 'SibSp', 'Parch'], inplace=True)

In [10]:
# Handle Missing Values
# Fill Age with median
df['Age'].fillna(df['Age'].median(), inplace=True)
# Fill Embarked with mode (most frequent value)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
# Fill Fare with median
df['Fare'].fillna(df['Fare'].median(), inplace=True)

/tmp/ipykernel_831/1491839309.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_831/1491839309.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

In [11]:
# Encode Categorical Data
label_encoder = LabelEncoder()
df['Sex'] = label_encoder.fit_transform(df['Sex'])          # male=1, female=0
df['Embarked'] = label_encoder.fit_transform(df['Embarked'])  # C=0, Q=1, S=2

In [12]:
print("\nPreprocessed Data:")
print(df.head())


Preprocessed Data:
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0       0.0     3.0    1  22.0    1.0    0.0   7.2500         2
1       1.0     1.0    0  38.0    1.0    0.0  71.2833         0
2       1.0     3.0    0  26.0    0.0    0.0   7.9250         2
3       1.0     1.0    0  35.0    1.0    0.0  53.1000         2
4       0.0     3.0    1  35.0    0.0    0.0   8.0500         2


In [13]:
print("\nMissing values after preprocessing:")
print(df.isnull().sum())


Missing values after preprocessing:
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [14]:
# Split Features and Target
X = df.drop(columns=['Survived'])
y = df['Survived']

In [15]:
# Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [17]:
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")


Training set size: 712
Testing set size: 179


# MODEL IMPLEMENTATION - Define Neural Network Architecture

In [18]:
model = Sequential([
    # Input layer + First hidden layer
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),

    # Second hidden layer
    Dense(32, activation='relu'),
    Dropout(0.2),

    # Third hidden layer
    Dense(16, activation='relu'),

    # Output layer (sigmoid for binary classification)
    Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
# Compile the Model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy'])

In [20]:
# Display model summary
print("\nModel Architecture:")
model.summary()


Model Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,137 (12.25 KB)

 Trainable params: 3,137 (12.25 KB)

 Non-trainable params: 0 (0.00 B)

# TRAIN THE MODEL

In [21]:
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 7s 230ms/step - accuracy: 0.5518 - loss: 0.6771 - val_accuracy: 0.7902 - val_loss: 0.6267
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.7311 - loss: 0.6101 - val_accuracy: 0.8322 - val_loss: 0.5480
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7610 - loss: 0.5582 - val_accuracy: 0.8252 - val_loss: 0.4819
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7838 - loss: 0.5125 - val_accuracy: 0.8322 - val_loss: 0.4349
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7873 - loss: 0.4854 - val_accuracy: 0.8322 - val_loss: 0.4119
Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7926 - loss: 0.4793 - val_accuracy: 0.8462 - val_loss: 0.3988
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7891 - loss: 0.4779 - val_accuracy: 0.8182 - val_loss: 0.4014
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7873 - loss: 0.4650 - val_accuracy: 0.8322

# EVALUATE THE MODEL

In [22]:
# Predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step


In [23]:
# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

In [24]:
print("\n" + "=" * 50)
print("MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


MODEL EVALUATION RESULTS
Accuracy:  0.8268
Precision: 0.8413
Recall:    0.7162
F1-Score:  0.7737


In [25]:
#  Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)


Confusion Matrix:
[[95 10]
 [21 53]]


In [26]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))


Classification Report:
              precision    recall  f1-score   support

Not Survived       0.82      0.90      0.86       105
    Survived       0.84      0.72      0.77        74

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.82       179



In [27]:
# Training History - Loss and Accuracy
print("\nTraining History (Last 5 Epochs):")
print(f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<14}{'Train Acc':<14}{'Val Acc'}")
for i in range(-5, 0):
    epoch_num = len(history.history['loss']) + i + 1
    print(f"{epoch_num:<8}{history.history['loss'][i]:<14.4f}{history.history['val_loss'][i]:<14.4f}"
          f"{history.history['accuracy'][i]:<14.4f}{history.history['val_accuracy'][i]:.4f}")


Training History (Last 5 Epochs):
Epoch   Train Loss    Val Loss      Train Acc     Val Acc
96      0.3905        0.4011        0.8471        0.8531
97      0.3930        0.3949        0.8436        0.8392
98      0.3738        0.3995        0.8348        0.8462
99      0.3668        0.3988        0.8506        0.8531
100     0.3739        0.3934        0.8506        0.8392


#  CONCLUSION

In [28]:
print("\n" + "=" * 50)
print("CONCLUSION")
print("=" * 50)
print(f"""
- The neural network achieved an accuracy of {accuracy:.2%} on the test set.
- The model uses 3 hidden layers (64, 32, 16 neurons) with ReLU activation.
- Dropout regularization was applied to reduce overfitting.
- Binary cross-entropy loss with Adam optimizer was used for training.

Potential Improvements:
1. Hyperparameter tuning (learning rate, batch size, epochs)
2. Feature engineering (e.g., extracting titles from names)
3. Using more complex architectures or ensemble models
4. Cross-validation for more robust evaluation
5. Early stopping to prevent overfitting
""")


CONCLUSION

- The neural network achieved an accuracy of 82.68% on the test set.
- The model uses 3 hidden layers (64, 32, 16 neurons) with ReLU activation.
- Dropout regularization was applied to reduce overfitting.
- Binary cross-entropy loss with Adam optimizer was used for training.

Potential Improvements:
1. Hyperparameter tuning (learning rate, batch size, epochs)
2. Feature engineering (e.g., extracting titles from names)
3. Using more complex architectures or ensemble models
4. Cross-validation for more robust evaluation
5. Early stopping to prevent overfitting



In [29]:
from sklearn.ensemble import RandomForestClassifier

In [30]:
rf = RandomForestClassifier(random_state=42)

In [31]:
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [32]:
y_pred = rf.predict(X_test)

In [33]:
accuracy_score(y_test,y_pred)

0.8212290502793296

In [34]:
confusion_matrix(y_test,y_pred)

array([[92, 13],
       [19, 55]])

In [35]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

         0.0       0.83      0.88      0.85       105
         1.0       0.81      0.74      0.77        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179

